Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Verification of Improved Ramsey Number Bounds

This notebook verifies the claims in the `improved_bounds` folder. For each file `R(r, s) >= n+1.txt`, we:
1. Verify that the graph has `n` vertices
2. Verify the graph contains no `r`-clique
3. Verify the complement graph contains no `s`-clique (equivalently, the graph has no independent set of size `s`)

In [ ]:
%pip install igraph

In [ ]:
import os
import re
import igraph

BOUNDS_DIR = 'improved_bounds'

all_passed = True

for filename in sorted(os.listdir(BOUNDS_DIR)):
    if not filename.endswith('.txt'):
        continue

    match = re.match(r'R\((\d+),\s*(\d+)\)\s*>=\s*(\d+)\.txt', filename)
    if not match:
        print(f'SKIP: Could not parse filename: {filename}')
        continue

    r = int(match.group(1))
    s = int(match.group(2))
    n_plus_1 = int(match.group(3))
    n = n_plus_1 - 1

    print(f'--- Verifying {filename} ---')
    print(f'  Claim: R({r}, {s}) >= {n_plus_1}')
    print(f'  Expected number of vertices: {n}')

    filepath = os.path.join(BOUNDS_DIR, filename)
    with open(filepath, 'r') as f:
        content = f.read()

    cleaned = content.replace('[', '').replace(']', '')
    values = list(map(int, cleaned.split()))
    side = int(len(values) ** 0.5)
    rows = [values[i * side:(i + 1) * side] for i in range(side)]

    num_vertices = side
    assert len(values) == side * side, \
        f'Adjacency matrix is not square: {len(values)} values for side {side}'

    if num_vertices != n:
        print(f'  FAIL: Graph has {num_vertices} vertices, expected {n}')
        all_passed = False
        continue
    print(f'  OK: Graph has {num_vertices} vertices')

    edges = [(i, j) for i in range(side) for j in range(i + 1, side) if rows[i][j] == 1]
    g = igraph.Graph(n=num_vertices, edges=edges, directed=False)

    r_cliques = g.cliques(min=r, max=r)
    if len(r_cliques) > 0:
        print(f'  FAIL: Found {len(r_cliques)} clique(s) of size {r}')
        all_passed = False
        continue
    print(f'  OK: No {r}-cliques found')

    g_complement = g.complementer(loops=False)

    s_cliques = g_complement.cliques(min=s, max=s)
    if len(s_cliques) > 0:
        print(f'  FAIL: Found {len(s_cliques)} independent set(s) of size {s}')
        all_passed = False
        continue
    print(f'  OK: No independent set of size {s} found')

    print(f'  PASSED ✓')
    print()

if all_passed:
    print('=' * 50)
    print('ALL CLAIMS VERIFIED SUCCESSFULLY ✓')
else:
    print('=' * 50)
    print('SOME CLAIMS FAILED ✗')
